# DeepForest finetuning


In [ ]:
!pip install -q deepforest datasets scikit-image scipy

In [ ]:
import os, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import xy as rasterio_xy
from rasterio.windows import Window
from pathlib import Path
from PIL import Image
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from sklearn.model_selection import train_test_split
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.filters import gaussian
from skimage.morphology import remove_small_objects
from skimage.measure import label, regionprops
import torch
from deepforest import main as df_main
from datasets import load_dataset
from pathlib import Path
import torch

TIF_PATH        = "/content/drive/MyDrive/Kopie souboru bologna_rgb_2023.tif"
WORK_DIR        = Path("/content/drive/MyDrive/3-30-300/data/Italy/Bologna/3trees")
MODEL_SAVE      = "/content/deepforest_tcd_eu 1.pt"
RAW_PREDS_CSV   = str(WORK_DIR / "raw_predictions_checkpoint.csv")
OUTPUT_CLEAN    = str(WORK_DIR / "tree_detections_clean.gpkg")
OUTPUT_REJECTED = str(WORK_DIR / "tree_detections_rejected.gpkg")

WORK_DIR.mkdir(parents=True, exist_ok=True)

# Training
MAX_TILES    = 2000
TRAIN_EPOCHS = 15
TRAIN_LR     = 1e-4
TRAIN_BATCH  = 4

# Inference
PATCH_SIZE         = 600
PATCH_OVERLAP      = 0.05   # keep low
IOU_THRESHOLD      = 0.2    # no speed cost
INFER_SCORE_THRESH = 0.2    # no speed cost

# Post-processing
SCORE_THRESH     = 0.2     # actual quality gate applied after inference
MIN_AREA_M2      = 2.0
MAX_AREA_M2      = 200.0
MAX_ASPECT_RATIO = 3.0
SIZE_SCORE_RULES = [
    (4.0,          0.40),
    (15.0,         0.25),
    (30.0,         0.20),
    (float('inf'), 0.15),
]

# NMS
NMS_IOU_THRESH  = 0.4
NMS_CELL_SIZE_M = 100


MERGE_OVERLAPPING    = False
TARGET_CRS           = None
DIAGNOSE_ONLY        = False
LAYER_NAME           = "tree_crowns"   # fixed quotes

TILES_DIR.mkdir(exist_ok=True)
print("Config OK")

In [ ]:
ds = load_dataset("restor/tcd", split="train", streaming=True)
european = (
    s for s in ds
    if 35 <= s["lat"] <= 72 and -25 <= s["lon"] <= 45
)

IMG_SIZE = 2048
rows = []

for i, sample in enumerate(european):
    if i >= MAX_TILES:
        break

    img_path = TILES_DIR / f"tile_{i:04d}.png"
    if not img_path.exists():
        sample["image"].save(img_path)

    raw = sample.get("coco_annotations")
    if not raw:
        continue
    try:
        annots = json.loads(raw) if isinstance(raw, str) else raw
    except Exception:
        continue

    if isinstance(annots, dict):
        annots = annots.get("annotations", [])

    for ann in annots:
        if ann.get("category_id") not in [1, 2]:  # 1=canopy, 2=individual tree
            continue
        if ann.get("iscrowd", 0):
            continue
        x, y, w, h = ann["bbox"]
        if w < 5 or h < 5:
            continue
        rows.append({
            "image_path": str(img_path),
            "xmin": float(x),
            "ymin": float(y),
            "xmax": float(x + w),
            "ymax": float(y + h),
            "label": "Tree",
        })

df = pd.DataFrame(rows)
assert "image_path" in df.columns and len(df) > 0, "Empty — check coco_annotations structure"

# Clip & drop tiny boxes
df[["xmin", "ymin"]] = df[["xmin", "ymin"]].clip(lower=0)
df["xmax"] = df["xmax"].clip(upper=IMG_SIZE)
df["ymax"] = df["ymax"].clip(upper=IMG_SIZE)
df = df[((df["xmax"] - df["xmin"]) >= 5) & ((df["ymax"] - df["ymin"]) >= 5)]

# Train / val split (tile-level)
tiles = df["image_path"].unique()
train_tiles, val_tiles = train_test_split(tiles, test_size=0.15, random_state=42)
df[df["image_path"].isin(train_tiles)].to_csv(TRAIN_CSV, index=False)
df[df["image_path"].isin(val_tiles)].to_csv(VAL_CSV,   index=False)

print(f"{len(df):,} boxes from {len(tiles)} tiles  "
      f"({len(train_tiles)} train / {len(val_tiles)} val)")

In [ ]:
model = df_main.deepforest()
model.load_model(model_name="weecology/deepforest-tree", revision="main")

# All config before create_trainer
model.config["train"]["csv_file"]        = TRAIN_CSV
model.config["train"]["root_dir"]        = str(TILES_DIR)
model.config["validation"]["csv_file"]   = VAL_CSV
model.config["validation"]["root_dir"]   = str(TILES_DIR)
model.config["train"]["epochs"]          = TRAIN_EPOCHS
model.config["train"]["lr"]              = TRAIN_LR
model.config["train"]["batch_size"]      = TRAIN_BATCH
model.config["train"]["augment"]         = True
model.config["score_thresh"]             = INFER_SCORE_THRESH
model.config["nms_thresh"]               = 0.15

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

model.create_trainer(fast_dev_run=False)
model.trainer.fit(model)
model.save_model(MODEL_SAVE)
print(f"Model saved to {MODEL_SAVE}")

In [ ]:
m = df_main.deepforest()
m.load_model(model_name="weecology/deepforest-tree", revision="main")
state = torch.load(MODEL_SAVE, map_location="cuda")
m.model.load_state_dict(state["state_dict"])
m.config["score_thresh"] = INFER_SCORE_THRESH
m.config["batch_size"] = 2
m.config["workers"] = 0
m.create_trainer()
print("Fine-tuned model loaded")

predictions = m.predict_tile(
    path=TIF_PATH,
    patch_size=PATCH_SIZE,
    patch_overlap=PATCH_OVERLAP,
    iou_threshold=IOU_THRESHOLD,
    dataloader_strategy="window",
)

predictions.to_csv(RAW_PREDS_CSV, index=False)
print(f"Saved {len(predictions):,} detections to {RAW_PREDS_CSV}")

predictions.head()

In [ ]:
df = pd.read_csv(RAW_PREDS_CSV)
df = df[df["score"] >= 0.3]
df = df[df["label"] == "Tree"]

# Pixel bbox → real-world geometry
with rasterio.open(TIF_PATH) as src:
    crs = src.crs
    transform = src.transform

def pixel_box_to_geom(row):
    x_min, y_min = xy(transform, row["ymin"], row["xmin"])
    x_max, y_max = xy(transform, row["ymax"], row["xmax"])
    return box(x_min, y_min, x_max, y_max)

df["geometry"] = df.apply(pixel_box_to_geom, axis=1)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=crs)
gdf.to_file(f"{WORK_DIR}/tree_detections.gpkg", driver="GPKG")
print(f"Saved {len(gdf):,} trees to {WORK_DIR}/tree_detections.gpkg")